# Mini Project 2: House Price Prediction Model
**SkillNexis ML & AI Internship, Week 2** (also covers *Assignment 1: Linear Regression on a housing dataset*)

**Dataset:** Kaggle, [House Prices: Advanced Regression Techniques](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data). Download `train.csv` and save it as `data/house_prices_train.csv`.

**Tasks**
1. Train a Linear Regression model
2. Predict house prices
3. Evaluate the R² score
4. Plot predicted vs actual values

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

RANDOM_STATE = 42
Path("images").mkdir(exist_ok=True)

## 1. Load and inspect the data

In [ ]:
DATA_PATH = Path("data/house_prices_train.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Download train.csv from Kaggle (House Prices) and save it as data/house_prices_train.csv"
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
# Target distribution and the columns with the most missing values
print(df["SalePrice"].describe())
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0].head(10)

## 2. Prepare features
* `Id` is only a row label, so it is dropped.
* Numeric columns: missing values filled with the **median**, then scaled.
* Categorical columns: missing values filled with the **most frequent** value, then one-hot encoded.
* All of this sits inside a `Pipeline`, so it is fitted on the training data only (no data leakage).

In [ ]:
X = df.drop(columns=["SalePrice", "Id"])
y = df["SalePrice"]

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()
print(f"{len(numeric_cols)} numeric features, {len(categorical_cols)} categorical features")

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric_cols),
    # min_frequency groups very rare categories together; rare dummy columns can make
    # plain Linear Regression produce extreme predictions on a few test rows
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10))]), categorical_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "Test:", X_test.shape)

## 3. Train the Linear Regression model

In [ ]:
lin_reg = Pipeline([("prep", preprocess), ("model", LinearRegression())])
lin_reg.fit(X_train, y_train)

## 4. Predict house prices and evaluate

In [ ]:
y_pred = lin_reg.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MSE : {mse:,.0f}")
print(f"RMSE: {rmse:,.0f}  (typical error in dollars)")
print(f"R²  : {r2:.4f}")

In [ ]:
# A few predictions next to the real prices
comparison = pd.DataFrame({"Actual": y_test.values, "Predicted": y_pred.round(0)}).head(10)
comparison["Error"] = comparison["Predicted"] - comparison["Actual"]
comparison

## 5. Plot predicted vs actual values
Points close to the red dashed line are accurate predictions.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, y_pred, alpha=0.6, edgecolor="k", linewidth=0.3)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, "r--", label="Perfect prediction")
ax.set_xlabel("Actual SalePrice")
ax.set_ylabel("Predicted SalePrice")
ax.set_title(f"Linear Regression: Predicted vs Actual (R² = {r2:.3f})")
ax.legend()
plt.tight_layout()
plt.savefig("images/predicted_vs_actual.png", dpi=150)
plt.show()

## 6. Bonus: compare with Random Forest
Week 2 also covers Random Forest, so here is how it compares on the same split.

In [ ]:
rf = Pipeline([("prep", preprocess),
               ("model", RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))])
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MSE": [mse, mean_squared_error(y_test, rf_pred)],
    "R²": [r2, r2_score(y_test, rf_pred)],
}).set_index("Model")
results

## 7. Conclusions
Fill these in from *your* output after running the notebook:
* R² of Linear Regression on the test set: **0.866316**
* Typical prediction error (RMSE): **32,022**
* Does Random Forest beat Linear Regression? **Yes**
* Where in the plot are the predictions worst (very expensive houses?): **The worst predictions occur in the upper-right region corresponding to very expensive houses. These points show the largest deviations from the diagonal (perfect prediction) line, indicating that the model tends to make larger errors for high-priced properties.**